# Imports

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from uavloc.data.rasters import load_raster
from uavloc.plotting.flight_video import FlightVideo, VideoConfig
from uavloc.simulation.arms import FilterParams, HeadSpec, build_correlator, estimator_factory
from uavloc.simulation.camera import CameraConfig, RasterCamera
from uavloc.simulation.route import sample_route
from uavloc.simulation.sequence import as_flight_result, load_flight_set, run_flights
from uavloc.track import REPO_ROOT, set_seed

# Params

In [ ]:
VIDEOS = [("bushcha-50p2422-25p8305", 1, "control"),      # locks on ~100 m along the village street
          ("rokety-50p2212-24p3470", 1, "control"),       # last minute over fields, every fix blind
          ("kryve-50p1899-23p8279", 1, "control"),        # lost early, re-acquired at the end (11.7 m)
          ("kryve-50p1899-23p8279", 1, "structure")]      # same route, never re-acquired
CONFIGS = {   # name: (segmenter, head, temperature)
    "control": ("experiments/033-class-loss-in-system/artifacts/segmenter-control/model-bn-satellite.pt",
                "experiments/033-class-loss-in-system/artifacts/head-control.pt", 0.5),
    "structure": ("experiments/033-class-loss-in-system/artifacts/segmenter-structure/model-bn-satellite.pt",
                  "experiments/033-class-loss-in-system/artifacts/head-structure.pt", 0.5),
}
LOSS_THRESHOLD_M = 20.0
KEYFRAME_OFFSETS_S = [-10.0, 0.0, 10.0]
VIDEO_FPS = 20.0
VIDEO_SIZE_PX = (1280, 720)
VIDEO_MAIN_PANEL = "map"
VIDEO_CORNER_PANELS = {"top-right": "drone", "bottom-right": "segmentation"}
VIDEO_MAP_RADIUS_M = 160.0
VIDEO_VIEW_WIDTH_PX = 640
VIDEO_STEP_STRIDE = 1
VIDEO_CODEC = "avc1"
SEED = 42
DEVICE = "cpu"                    # CPU-only process (bug-121)
ROUTE_SET = "sim-flights"
SPLIT = "test"
INITIAL_ERROR_M = 0.0
SIGMA0_M = 1.0
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp", "seg_channels": "all", "drone_seg_gsd_m": 0.385,
    "match_gsd_m": 0.385, "matching": "weighted-ncc", "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5],
    "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0, "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
FILTER = "particles"
Q_POS_M_PER_SQRT_S = 2.0
N_PARTICLES = 2000
K_MAX_HYPOTHESES = 5
PEAKS_PER_HYPOTHESIS = 3
PEAK_NMS_M = 10.0
MERGE_DISTANCE_M = 5.0
CHILD_FLOOR_SIGMA_M = 1.0
WINDOW_SIGMAS = 3.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
HEAD_ENCODER_CHANNELS = (32, 32, 16)
HEAD_REFINE_CHANNELS = 16
CAMERA_ASPECT_W_OVER_H = 16 / 9
VIDEO_DIR = Path("artifacts/videos")
KEYFRAME_DIR = Path("figures/videos")

# Data

In [ ]:
set_seed(SEED)
set_config, all_specs = load_flight_set(ROUTE_SET, split=SPLIT)
specs = {(s.village, s.route_index): s for s in all_specs}
VIDEO_DIR.mkdir(parents=True, exist_ok=True), KEYFRAME_DIR.mkdir(parents=True, exist_ok=True)

# Model

In [ ]:
def filter_params(temperature: float) -> FilterParams:
    return FilterParams(temperature, Q_POS_M_PER_SQRT_S, N_PARTICLES, K_MAX_HYPOTHESES, PEAKS_PER_HYPOTHESIS,
                        PEAK_NMS_M, MERGE_DISTANCE_M, CHILD_FLOOR_SIGMA_M, WINDOW_SIGMAS, WINDOW_CAP_HALF_M,
                        WINDOW_MARGIN_M)

In [ ]:
def replay_one(name: str, spec) -> tuple:
    segmenter, head, temperature = CONFIGS[name]
    correlator_config = {**CORRELATOR, "seg_checkpoint_drone": segmenter, "seg_checkpoint_sat": segmenter}
    head_spec = HeadSpec(REPO_ROOT / head, HEAD_ENCODER_CHANNELS, HEAD_REFINE_CHANNELS)
    built = {}

    def build(raster):
        built["correlator"] = build_correlator(raster, correlator_config, DEVICE, head_spec)
        return built["correlator"]

    factories = {FILTER: estimator_factory(FILTER, filter_params(temperature))}
    outcome = run_flights(set_config, [spec], INITIAL_ERROR_M, SIGMA0_M, build, factories)[FILTER][0]
    return outcome, built["correlator"]

In [ ]:
def first_loss_s(track) -> float | None:
    lost = track[track["est_error_m"] > LOSS_THRESHOLD_M]
    return None if lost.empty else float(lost["t_s"].iloc[0])

# Run

In [ ]:
video_config = VideoConfig(fps=VIDEO_FPS, frame_size_px=VIDEO_SIZE_PX, main_panel=VIDEO_MAIN_PANEL,
                           corner_panels=VIDEO_CORNER_PANELS, map_mode="follow", map_radius_m=VIDEO_MAP_RADIUS_M,
                           segmentation_source="every-frame", step_stride=VIDEO_STEP_STRIDE, codec=VIDEO_CODEC)
camera_config = replace(CameraConfig(set_config.fov_w_deg), image_width_px=VIDEO_VIEW_WIDTH_PX)

In [ ]:
made = []
for village, route_index, name in VIDEOS:
    spec = specs[(village, route_index)]
    outcome, correlator = replay_one(name, spec)
    poses = sample_route(spec.waypoints, set_config.speed_mps, set_config.step_s, set_config.altitude_m)
    result = as_flight_result(outcome, poses, name)
    camera = RasterCamera(load_raster({"kind": "multidate", "place": village, "date": set_config.drone_date}), camera_config)
    segment = lambda view, w_m, h_m, c=correlator: c.features.drone_probabilities(view, w_m, h_m).argmax(0)
    map_raster = load_raster({"kind": "multidate", "place": village, "date": set_config.map_date})
    video = FlightVideo(result, camera, map_raster, video_config, segment)
    stem = f"{village.split('-')[0]}-{route_index}-{name}"
    video.save(VIDEO_DIR / f"{stem}.mp4")
    made.append({"stem": stem, "video": video, "track": outcome.track, "end_m": outcome.track["est_error_m"].iloc[-1],
                 "first_loss_s": first_loss_s(outcome.track)})
[(m["stem"], round(m["end_m"], 1), m["first_loss_s"]) for m in made]

# Plots

In [ ]:
for m in made:
    if m["first_loss_s"] is None:
        continue
    times = m["track"]["t_s"].to_numpy()
    steps = [int(np.clip(np.searchsorted(times, m["first_loss_s"] + dt), 0, len(times) - 1)) for dt in KEYFRAME_OFFSETS_S]
    fig, axes = plt.subplots(1, len(steps), figsize=(7 * len(steps), 4.2))
    for ax, step in zip(axes, steps):
        frame = m["video"].frame(step)
        Image.fromarray(frame).save(KEYFRAME_DIR / f"{m['stem']}-t{times[step]:.0f}s.jpg", quality=88)
        ax.imshow(frame), ax.axis("off")
        ax.set_title(f"{m['stem']}  t = {times[step]:.0f} s")
    fig.tight_layout()